# Compare FD map construction choices

Fit one interleaving between a Y graph and an upside-down Y. Reuse that interleaving and the same seed to compare point selection and path rules.

In [1]:
import sys
from pathlib import Path

# Make project imports available when Jupyter starts in experiment/.
if Path.cwd().name == "experiment":
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import display
from cereeberus import Interleave, MapperGraph
from core.distortion import functional_distortion
from core.fd_map_build import maps_from_interleave


## 1. Fit the interleaving

In [2]:
def make_graph(nodes, edges):
    M = MapperGraph()
    for name, h in nodes:
        M.add_node(name, h)
    for u, v in edges:
        M.add_edge(u, v)
    return M


def _y_data():
    YF_n = [("f0", 0), ("f1", 1), ("f2", 2), ("j", 3), ("L4", 4), ("L5", 5), ("R4", 4)]
    YF_e = [("f0", "f1"),("f1", "f2"),("f2", "j"),("j", "L4"),("L4", "L5"),("j", "R4")]
    YG_n = [("g5", 5), ("g4", 4), ("g3", 3), ("q", 2), ("b1", 1), ("b0", 0), ("c1", 1)]
    YG_e = [("g5", "g4"),("g4", "g3"),("g3", "q"),("q", "b1"),("b1", "b0"),("q", "c1")]
    
    return (YF_n, YF_e, YG_n, YG_e)

F_nodes, F_edges, G_nodes, G_edges = _y_data()
mapper_F = make_graph(F_nodes, F_edges)
mapper_G = make_graph(G_nodes, G_edges)


In [3]:
# fit the interleaving
interleaving = Interleave(mapper_F, mapper_G)
print("Interleaving parameter:", interleaving.fit())


Interleaving parameter: 1


## 2. Compare construction choices

- `points` controls the selection of vertex images and handoff points within the allowed sets. `random` samples among the nearest-height candidates. `close` (the default) also scores route spans, and `joint` scores the two directions together using vertex round trips.
- `paths` chooses the route inside each allowed set. `fewest_edges` favors short walks, while `min_span` (the default) uses one shared height window for the entire route through the handoff. It minimizes the width first, then the total edge count.
- `seed` controls random choices.

Compare the first two rows for the effect of the path rule. Compare the baseline, random, and round-trip rows for the effect of point selection. All scores use the same evaluator.

In [4]:
map_seed = 100
choices = [
    ("Baseline", "close", "min_span"),
    ("Change path rule", "close", "fewest_edges"),
    ("Random point selection", "random", "min_span"),
    ("Score round trips", "joint", "min_span"),
]
runs = {}
rows = []
for label, points, paths in choices:
    maps = maps_from_interleave(
        interleaving, seed=map_seed, points=points, paths=paths
    )
    result = functional_distortion(*(maps[key] for key in ("F", "G", "Phi", "Psi")), decimals=3)
    runs[label] = maps, result
    rows.append({"Setting": label, "points": points, "paths": paths,
                 "Height error Φ": result["height_Phi_units"],
                 "Height error Ψ": result["height_Psi_units"],
                 "D": result["D_units"], "FD": result["FD_units"]})
table = pd.DataFrame(rows).set_index("Setting")
display(table)
best_label = min(runs, key=lambda label: runs[label][1]["FD_units"])
print(f"Lowest FD among these runs: {runs[best_label][1]['FD_units']}")


,points,paths,Height error Φ,Height error Ψ,D,FD
Setting,,,,,,
Baseline,close,min_span,1.0,0.0,1.0,1.0
Change path rule,close,fewest_edges,1.0,0.0,1.0,1.0
Random point selection,random,min_span,1.0,0.0,1.0,1.0
Score round trips,joint,min_span,1.0,0.0,1.0,1.0


Lowest FD among these runs: 1.0
